# **RAG Pipeline — Machine Learning Study Assistant**

This notebook builds and evaluates a Retrieval-Augmented Generation (RAG) pipeline
grounded in:
- **Machine Learning lecture notes** (Classical ML: Regression, Classification, Clustering)
- **scikit-learn documentation** (Linear Models, Logistic Regression, KNN, Decision Trees, Ensembles, SVM, Clustering)
- **pandas documentation** (data cleaning, indexing, groupby)


## **Setup**

In [1]:
# !pip install pypdf sentence-transformers chromadb ollama pandas numpy -q --break-system-packages


In [2]:
import os
import re
import json
import glob
from pypdf import PdfReader
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw_documents"
VECTOR_STORE_DIR = "../data/vector_store"
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)


## **Load & Inspect**



In [3]:
NAV_LINE_PATTERNS = [
    r"^Section Navigation$",
    r"^Collapse Sidebar$",
    r"^On this page$",
    r"^\d+(\.\d+)*\.\s.+",        # numbered TOC entries like "1.1. Linear Models"
    r"^(Install|User Guide|API|Examples|Community|More)$",
    r"^\d+\.\d+(\.\d+)?\s*\(stable\)$",
    r"^©\s*Copyright.*",
    r"^Previous$", r"^Next$",
    r"^scikit-learn is financially supported.*",
    r"^Enterprise-grade solutions.*",
]
_compiled_nav_patterns = [re.compile(p) for p in NAV_LINE_PATTERNS]


def clean_page_text(raw_text):
    """Strip browser-print boilerplate (sidebar TOC, nav buttons, footer) line by line,
    keeping only the actual documentation content. This matters because scikit-learn/pandas
    pages were exported via 'Print to PDF' and carry their sidebar navigation on every page,
    which otherwise pollutes chunks and hurts retrieval precision"""
    lines = raw_text.split("\n")
    kept = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if any(p.match(stripped) for p in _compiled_nav_patterns):
            continue
        kept.append(stripped)
    return " ".join(kept)


def load_pdf_pages(path):
    """Extract text per page from a PDF, cleaned of navigation boilerplate"""
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        gallery_marker = "\nGallery examples\n"          
        if gallery_marker in text:                       
            text = text.split(gallery_marker)[0]  
        text = clean_page_text(text)
        text = re.sub(r"\s+", " ", text).strip()
        pages.append({"page_num": i + 1, "text": text})
    return pages


def discover_pdfs(data_dir):
    return sorted(glob.glob(os.path.join(data_dir, "**", "*.pdf"), recursive=True))


pdf_paths = discover_pdfs(DATA_DIR)
print(f"Found {len(pdf_paths)} PDF file(s):")
for p in pdf_paths:
    print(" -", p)

Found 18 PDF file(s):
 - ../data/raw_documents/lecture_notes/ML papers (Full Source).pdf
 - ../data/raw_documents/pandas_docs/Categorical data — pandas documentation.pdf
 - ../data/raw_documents/pandas_docs/Essential basic functionality — pandas documentation.pdf
 - ../data/raw_documents/pandas_docs/Group by — pandas documentation.pdf
 - ../data/raw_documents/pandas_docs/Indexing and selecting data — pandas documentation.pdf
 - ../data/raw_documents/pandas_docs/Merge, join, concatenate and compare — pandas documentation.pdf
 - ../data/raw_documents/pandas_docs/Reshaping and pivot tables — pandas documentation.pdf
 - ../data/raw_documents/pandas_docs/Working with missing data — pandas documentation.pdf
 - ../data/raw_documents/sklearn_docs/Clustering — scikit-learn documentation.pdf
 - ../data/raw_documents/sklearn_docs/Cross-validation — scikit-learn documentation.pdf
 - ../data/raw_documents/sklearn_docs/Decision Trees — scikit-learn documentation.pdf
 - ../data/raw_documents/sklearn_

In [4]:
inspection_rows = []
all_docs = {}  # filename -> list of page dicts

for path in pdf_paths:
    filename = os.path.basename(path)
    try:
        pages = load_pdf_pages(path)
        all_docs[filename] = pages
        n_pages = len(pages)
        empty_pages = sum(1 for p in pages if len(p["text"]) < 20)
        total_chars = sum(len(p["text"]) for p in pages)
        status = "OK" if empty_pages < n_pages * 0.3 else "NEEDS OCR / CHECK"
        inspection_rows.append([filename, n_pages, empty_pages, total_chars, status])
    except Exception as e:
        inspection_rows.append([filename, "ERROR", "-", "-", str(e)])

inspection_df = pd.DataFrame(
    inspection_rows,
    columns=["file", "pages", "pages_with_little_text", "total_chars", "status"]
)
inspection_df


Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 170 0 (offset 0)
Ignoring wrong pointing object 244 0 (offset 0)
Ignoring wrong pointing object 245 0 (offset 0)
Ignoring wrong pointing object 246 0 (offset 0)
Ignoring wrong pointing object 247 0 (offset 0)
Ignoring wrong pointing object 248 0 (offset 0)
Ignoring wrong pointing object 249 0 (offset 0)
Ignoring wrong pointing object 250 0 (offset 0)
Ignoring wrong pointing object 251 0 (offset 0)
Ignoring wrong pointing object 252 0 (offset 0)
Ignoring wrong pointing object 253 0 (offset 0)
Ignoring wrong pointing object 254 0 (offset 0)
Ignoring wrong pointing object 255 0 (offset 0)
Ignoring wrong pointing object 340 0 (offset 0)
Ignoring wrong pointing object 341 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 364 0 (offset 0)
Ignoring wrong pointing object 517 0 (offset 0)
Ignoring wrong pointing object 548 0 (offset 0)
Ignoring wrong pointing object 583 0 (offset

,file,pages,pages_with_little_text,total_chars,status
0,ML papers (Full Source).pdf,120,1,229455,OK
1,Categorical data — pandas documentation.pdf,3,0,41920,OK
2,Essential basic functionality — pandas documen...,6,0,95230,OK
3,Group by — pandas documentation.pdf,4,0,65931,OK
4,Indexing and selecting data — pandas documenta...,5,0,83168,OK
5,"Merge, join, concatenate and compare — pandas ...",3,0,31180,OK
6,Reshaping and pivot tables — pandas documentat...,2,0,28459,OK
7,Working with missing data — pandas documentati...,2,0,24304,OK
8,Clustering — scikit-learn documentation.pdf,3,0,57210,OK
9,Cross-validation — scikit-learn documentation.pdf,2,0,32495,OK


**Observations:**

- The dataset contains 18 PDF documents covering Machine Learning, scikit-learn, and pandas topics.
- All documents were successfully parsed using `pypdf`.
- The documents are text-based PDFs, so no OCR step was required.
- The main lecture notes contain 120 pages and cover the core Machine Learning topics.
- One page had relatively little extracted text because it contains mainly graphical content, but the document remained usable for text-based retrieval.
- The scikit-learn and pandas documents were exported from documentation pages using "Print to PDF", so some navigation elements and graphical content required cleaning before chunking.

## **Chunking Strategy**

**Justification:**
- 800 characters (~120-150 words) is large enough to keep a full definition, formula
  explanation, or code example intact (important for technical docs like scikit-learn/pandas),
  while still being small enough for precise retrieval — a chunk that's too large dilutes
  relevance (a match might be based on one sentence buried in a huge chunk).
- A 150-character overlap (~19%) reduces the chance that a concept explained across a chunk
  boundary gets split awkwardly, without duplicating so much text that the vector store bloats.
- We chunk **per page** rather than across the whole document so that every chunk keeps a single,
  accurate page number for citation — required by Phase 2.4.
- We considered a section-based strategy (splitting on numbered headers like "1.1.1 Ordinary
  Least Squares"), which would be more semantically clean, but headers are not formatted
  consistently across the lecture notes and the scikit-learn/pandas PDFs, making reliable
  header-detection fragile within the project's time budget. Fixed-size chunking is more robust
  across heterogeneous sources.





In [5]:
def chunk_pages(pages, source, chunk_size=800, overlap=150):
    """Split each page's text into overlapping fixed-size chunks"""
    chunks = []
    for page in pages:
        text = page["text"]
        if len(text) < 30:
            continue  # skip near-empty pages (covers, blank pages)
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_str = text[start:end].strip()
            if len(chunk_str) > 50:
                chunks.append({
                    "text": chunk_str,
                    "source": source,
                    "page": page["page_num"],
                })
            if end >= len(text):
                break
            start += chunk_size - overlap
    return chunks


all_chunks = []
for filename, pages in all_docs.items():
    all_chunks.extend(chunk_pages(pages, source=filename))

print(f"Total chunks created: {len(all_chunks)}")
avg_len = np.mean([len(c["text"]) for c in all_chunks])
print(f"Average chunk length: {avg_len:.0f} characters")

# quick per-source breakdown
chunk_counts = pd.Series([c["source"] for c in all_chunks]).value_counts()
chunk_counts


Total chunks created: 1567
Average chunk length: 764 characters


ML papers (Full Source).pdf                                        387
Essential basic functionality — pandas documentation.pdf           149
Metrics and scoring — scikit-learn documentation.pdf               146
Indexing and selecting data — pandas documentation.pdf             129
Group by — pandas documentation.pdf                                103
Clustering — scikit-learn documentation.pdf                         89
Ensembles — scikit-learn d.pdf                                      81
Categorical data — pandas documentation.pdf                         66
Linear Models — scikit-learn documentation.pdf                      65
Preprocessing data — scikit-learn documentation.pdf                 60
Cross-validation — scikit-learn documentation.pdf                   51
Merge, join, concatenate and compare — pandas documentation.pdf     49
Reshaping and pivot tables — pandas documentation.pdf               44
Nearest Neighbors — scikit-learn documentation.pdf                  39
Workin

In [6]:
# Sanity check
for c in all_chunks[:2]:
    print(f"[{c['source']} - page {c['page']}]")
    print(c["text"][:300])
    print("-" * 80)


[ML papers (Full Source).pdf - page 1]
MACHINE LEARNING [R17A0534] LECTURE NOTES B.TECH IV YEAR – I SEM(R17) (2020-21) DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY (Autonomous Institution – UGC, Govt. of India) Recognized under 2(f) and 12 (B) of UGC ACT 1956 (Affiliated to JNTUH, Hyderab
--------------------------------------------------------------------------------
[ML papers (Full Source).pdf - page 2]
IV Year B. Tech. CSE –II Sem L T/P/D C 4 1/- / - 3 (R17A0534) Machine Learning Objectives:  Acquire theoretical Knowledge on setting hypothesis for pattern recognition.  Apply suitable machine learning techniques for data handling and to gain knowledge from it.  Evaluate the performance of algori
--------------------------------------------------------------------------------


## **Embeddings & Vector Store**

We embed every chunk using `sentence-transformers/all-MiniLM-L6-v2` (fast, strong general-purpose
embedding model, 384 dimensions) and persist the vectors in a local **ChromaDB** collection so the
FastAPI backend can load it directly without recomputing embeddings at request time.


In [7]:
from sentence_transformers import SentenceTransformer
import chromadb

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10488.81it/s]


Embedding model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


/var/folders/8b/7mhs_7j12dzblvz_fw7n87zr0000gn/T/ipykernel_2769/3461395990.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [8]:
# Persistent Chroma client -> writes directly to the folder the backend will load from
client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

COLLECTION_NAME = "ml_rag_assistant"

# Fresh start each run (safe for repeated notebook execution)
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(name=COLLECTION_NAME)
print("Created collection:", COLLECTION_NAME)


Created collection: ml_rag_assistant


In [9]:
BATCH_SIZE = 64

texts = [c["text"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
metadatas = [{"source": c["source"], "page": c["page"]} for c in all_chunks]

for start in range(0, len(texts), BATCH_SIZE):
    end = start + BATCH_SIZE
    batch_texts = texts[start:end]
    batch_embeddings = embedding_model.encode(batch_texts, show_progress_bar=False).tolist()
    collection.add(
        ids=ids[start:end],
        embeddings=batch_embeddings,
        documents=batch_texts,
        metadatas=metadatas[start:end],
    )

print(f"Indexed {collection.count()} chunks into ChromaDB at '{VECTOR_STORE_DIR}'")


Indexed 1567 chunks into ChromaDB at '../data/vector_store'


##  **Retrieval & Prompting**

We implement a `retrieve()` function on top of the Chroma collection, test it against at least
10 sample questions covering Regression, Classification, Clustering, and library usage
(scikit-learn / pandas), then build a prompt template that grounds the LLM's answer in the
retrieved chunks and cites their source + page.


In [10]:
def retrieve(query, k=6):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        retrieved.append({
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "distance": dist,
        })
    return retrieved


In [11]:
sample_questions = [
    "What is the difference between Ridge and Lasso regression?",
    "How does K-means clustering work?",
    "What is logistic regression used for?",
    "How does the K-Nearest Neighbors algorithm classify a new point?",
    "What is a Random Forest and how does it relate to bagging?",
    "How do you handle missing data in pandas?",
    "What does the groupby function do in pandas?",
    "What is the difference between supervised and unsupervised learning?",
    "What is the purpose of a Support Vector Machine?",
    "How do you select rows and columns in a pandas DataFrame?",
]

for q in sample_questions:
    print(f"\nQ: {q}")
    for r in retrieve(q, k=2):
        print(f"  [{r['source']} p.{r['page']}] (dist={r['distance']:.3f}) {r['text'][:100]}...")



Q: What is the difference between Ridge and Lasso regression?
  [Linear Models — scikit-learn documentation.pdf p.1] (dist=0.883) is a constant and is the -norm of the coefficient vector. The implementation in the class Lasso uses...
  [Linear Models — scikit-learn documentation.pdf p.1] (dist=0.905) Squares by imposing a penalty on the size of the coefficients. The ridge coefficients minimize a pen...

Q: How does K-means clustering work?
  [ML papers (Full Source).pdf p.61] (dist=0.444) sian Logic, etc. It includes various algorithms such as Clustering, KNN, and Apriori algorithm. k-me...
  [Clustering — scikit-learn documentation.pdf p.1] (dist=0.499) een data. Inductive Clustering: An example of an inductive clustering model for handling new data. T...

Q: What is logistic regression used for?
  [Linear Models — scikit-learn documentation.pdf p.2] (dist=0.715) stic regression is implemented in LogisticRegression. Despite its name, it is implemented as a linea...
  [Linear Models —

In [12]:
PROMPT_TEMPLATE = """You are a helpful Machine Learning study assistant.
Answer the question using ONLY the context below. If the context does not contain
the answer, say you don't have enough information instead of guessing.

Context:
{context}

Question: {question}

Answer (cite the source file and page for each claim, like [source, p.X]):"""


def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(
        f"[{c['source']}, p.{c['page']}]\n{c['text']}" for c in retrieved_chunks
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)


# Preview a built prompt
example_chunks = retrieve(sample_questions[0], k=3)
print(build_prompt(sample_questions[0], example_chunks))


You are a helpful Machine Learning study assistant.
Answer the question using ONLY the context below. If the context does not contain
the answer, say you don't have enough information instead of guessing.

Context:
[Linear Models — scikit-learn documentation.pdf, p.1]
is a constant and is the -norm of the coefficient vector. The implementation in the class Lasso uses coordinate descent as the algorithm to fit the coefficients. See Least Angle Regression for another implementation: The function lasso_path is useful for lower-level tasks, as it computes the coefficients along the full path of possible values. L1-based models for Sparse Signals Compressive sensing: tomography reconstruction with L1 prior (Lasso) Common pitfalls in the interpretation of coefficients of linear models Lasso model selection: AIC-BIC / cross-validation Feature selection with Lasso As the Lasso regression yields sparse models, it can thus be used to perform feature selection, as detailed in L1-based feature sel

## **Evaluation**




In [13]:
import ollama

OLLAMA_MODEL = "llama3.2"

def generate_answer(question, k=6):
    retrieved = retrieve(question, k=k)
    prompt = build_prompt(question, retrieved)
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = response["message"]["content"]
    sources = [f"{c['source']} (p.{c['page']})" for c in retrieved]
    return answer, sources, retrieved


In [14]:
evaluation_rows = []

for q in sample_questions:
    answer, sources, retrieved = generate_answer(q)
    evaluation_rows.append({
        "question": q,
        "retrieved_sources": "; ".join(sources),
        "answer": answer,
        "correct": ""  # fill in manually after reading the answer: yes / no / partial
    })

eval_df = pd.DataFrame(evaluation_rows)
eval_df


,question,retrieved_sources,answer,correct
0,What is the difference between Ridge and Lasso...,Linear Models — scikit-learn documentation.pdf...,"According to the context, the difference betwe...",
1,How does K-means clustering work?,ML papers (Full Source).pdf (p.61); Clustering...,"To answer your question, I'll provide a step-b...",
2,What is logistic regression used for?,Linear Models — scikit-learn documentation.pdf...,"Based on the context provided, here are the an...",
3,How does the K-Nearest Neighbors algorithm cla...,ML papers (Full Source).pdf (p.82); Nearest Ne...,"Based on the provided context, here's how the ...",
4,What is a Random Forest and how does it relate...,ML papers (Full Source).pdf (p.72); Ensembles ...,Here are the answers to your questions:\n\n1. ...,
5,How do you handle missing data in pandas?,Working with missing data — pandas documentati...,"To handle missing data in pandas, the followin...",
6,What does the groupby function do in pandas?,Group by — pandas documentation.pdf (p.1); Gro...,"According to the pandas documentation, here's ...",
7,What is the difference between supervised and ...,ML papers (Full Source).pdf (p.58); ML papers ...,"Based on the provided context, here are the di...",
8,What is the purpose of a Support Vector Machine?,ML papers (Full Source).pdf (p.54); Support Ve...,The purpose of a Support Vector Machine (SVM) ...,
9,How do you select rows and columns in a pandas...,Group by — pandas documentation.pdf (p.4); Ind...,To select rows and columns in a pandas DataFra...,




**Main failure case observed:**
- Generic questions retrieved fragmented text because scikit-learn/pandas pages (saved as "Print to PDF") had sidebar navigation and footer text mixed into the actual content.
- Mitigation: added a clean_page_text() step to strip this boilerplate before chunking, and increased retrieved chunks (k) from 4 to 6. After the fix, the same question returned an accurate, well-structured answer.

**Additional failure case:** 
- Broad, generic questions (e.g., "steps of data 
preprocessing") sometimes retrieved chunks from an unrelated document (pandas 
groupby) due to semantic overlap in generic terms. However, specific technical 
queries naming the exact class (e.g., "What does StandardScaler do?") reliably 
retrieved the correct source, confirming the document was indexed correctly — 
query specificity, not data coverage, was the limiting factor here.


In [15]:
eval_df.to_csv("evaluation_results.csv", index=False)
print("Saved evaluation_results.csv")


Saved evaluation_results.csv


In [16]:
eval_df.loc[0, "correct"] = "partial"  
eval_df.loc[4, "correct"] = "partial"  
eval_df.loc[eval_df["correct"] == "", "correct"] = "yes"

eval_df.to_csv("evaluation_results.csv", index=False)
eval_df

,question,retrieved_sources,answer,correct
0,What is the difference between Ridge and Lasso...,Linear Models — scikit-learn documentation.pdf...,"According to the context, the difference betwe...",partial
1,How does K-means clustering work?,ML papers (Full Source).pdf (p.61); Clustering...,"To answer your question, I'll provide a step-b...",yes
2,What is logistic regression used for?,Linear Models — scikit-learn documentation.pdf...,"Based on the context provided, here are the an...",yes
3,How does the K-Nearest Neighbors algorithm cla...,ML papers (Full Source).pdf (p.82); Nearest Ne...,"Based on the provided context, here's how the ...",yes
4,What is a Random Forest and how does it relate...,ML papers (Full Source).pdf (p.72); Ensembles ...,Here are the answers to your questions:\n\n1. ...,partial
5,How do you handle missing data in pandas?,Working with missing data — pandas documentati...,"To handle missing data in pandas, the followin...",yes
6,What does the groupby function do in pandas?,Group by — pandas documentation.pdf (p.1); Gro...,"According to the pandas documentation, here's ...",yes
7,What is the difference between supervised and ...,ML papers (Full Source).pdf (p.58); ML papers ...,"Based on the provided context, here are the di...",yes
8,What is the purpose of a Support Vector Machine?,ML papers (Full Source).pdf (p.54); Support Ve...,The purpose of a Support Vector Machine (SVM) ...,yes
9,How do you select rows and columns in a pandas...,Group by — pandas documentation.pdf (p.4); Ind...,To select rows and columns in a pandas DataFra...,yes


## **Export**

The vector store is already persisted directly to `data/vector_store/` (via
`chromadb.PersistentClient`), so the backend can load it without rebuilding. We additionally
save the pipeline configuration (chunk size, overlap, embedding model name, collection name)
so the backend's `retrieval.py` service can load everything consistently.


In [17]:
config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "chunk_size": 800,
    "chunk_overlap": 150,
    "vector_store_path": VECTOR_STORE_DIR,
    "num_chunks_indexed": collection.count(),
}

with open(os.path.join(VECTOR_STORE_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Saved config:")
print(json.dumps(config, indent=2))


Saved config:
{
  "embedding_model": "all-MiniLM-L6-v2",
  "collection_name": "ml_rag_assistant",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "vector_store_path": "../data/vector_store",
  "num_chunks_indexed": 1567
}




**Next step**: Copy (or symlink) the `data/vector_store/` folder into `backend/data/vector_store/` so the
FastAPI backend can load it at startup
